In [0]:

source_path = "/Volumes/gpo_analytics/bronze/source_files/"

files = dbutils.fs.ls(source_path)

for file in files:
    print(file.name, "-", file.size, "bytes")


AdverseEvents.csv - 77024 bytes
DrugDispensation.csv - 229660 bytes
Investigator.csv - 5635 bytes
LabResults.csv - 551007 bytes
Randomization.csv - 73993 bytes
Shipment.csv - 46338 bytes
Site.csv - 8166 bytes
Study.csv - 2130 bytes
Subject.csv - 240446 bytes
Visit.csv - 677563 bytes


In [0]:

from pyspark.sql import functions as F
from pathlib import Path

catalog = "gpo_analytics"
schema = "bronze"
source_path = "/Volumes/gpo_analytics/bronze/source_files/"

csv_files = [
    f for f in dbutils.fs.ls(source_path)
    if f.name.lower().endswith(".csv")
]

for file in csv_files:
    table_name = Path(file.name).stem.lower()
    full_table_name = f"{catalog}.{schema}.{table_name}"

    print(f"Loading {file.name} into {full_table_name}")

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "FAILFAST")
        .csv(file.path)
        .withColumn("_source_file", F.lit(file.name))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

    print(f"Loaded {spark.table(full_table_name).count()} rows")
    
print("Bronze ingestion complete.")


Loading AdverseEvents.csv into gpo_analytics.bronze.adverseevents
Loaded 1200 rows
Loading DrugDispensation.csv into gpo_analytics.bronze.drugdispensation
Loaded 4500 rows
Loading Investigator.csv into gpo_analytics.bronze.investigator
Loaded 100 rows
Loading LabResults.csv into gpo_analytics.bronze.labresults
Loaded 10000 rows
Loading Randomization.csv into gpo_analytics.bronze.randomization
Loaded 1915 rows
Loading Shipment.csv into gpo_analytics.bronze.shipment
Loaded 600 rows
Loading Site.csv into gpo_analytics.bronze.site
Loaded 100 rows
Loading Study.csv into gpo_analytics.bronze.study
Loaded 20 rows
Loading Subject.csv into gpo_analytics.bronze.subject
Loaded 3000 rows
Loading Visit.csv into gpo_analytics.bronze.visit
Loaded 14398 rows
Bronze ingestion complete.


In [0]:

tables = [
    "study",
    "site",
    "subject",
    "visit",
    "adverseevents",
    "labresults",
    "drugdispensation",
    "randomization",
    "shipment",
    "investigator"
]

for table_name in tables:
    full_name = f"gpo_analytics.bronze.{table_name}"
    count = spark.table(full_name).count()
    print(f"{table_name}: {count:,} rows")


study: 20 rows
site: 100 rows
subject: 3,000 rows
visit: 14,398 rows
adverseevents: 1,200 rows
labresults: 10,000 rows
drugdispensation: 4,500 rows
randomization: 1,915 rows
shipment: 600 rows
investigator: 100 rows


In [0]:

# 1. Define the catalog and schema where our Bronze tables are stored
catalog_name = "gpo_analytics"
schema_name = "bronze"

# 2. List the nine tables we want to inspect
tables = [
    "site",
    "subject",
    "visit",
    "drugdispensation",
    "randomization",
    "shipment",
    "investigator",
    "labresults",
    "adverseevents"
]

# 3. Inspect each table one by one
for table_name in tables:

    # Build the full table name
    full_table_name = f"{catalog_name}.{schema_name}.{table_name}"

    print("=" * 80)
    print(f"TABLE: {full_table_name}")
    print("=" * 80)

    # Load the table as a Spark DataFrame
    df = spark.table(full_table_name)

    # Display column names and data types
    print("\nSCHEMA:")
    df.printSchema()

    # Count the total number of rows
    print(f"TOTAL ROWS: {df.count():,}")

    # Display the first five rows
    print("\nSAMPLE DATA:")
    display(df.limit(5))


TABLE: gpo_analytics.bronze.site

SCHEMA:
root
 |-- SiteID: string (nullable = true)
 |-- SiteName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- StudyID: string (nullable = true)
 |-- SiteStatus: string (nullable = true)
 |-- ActivationDate: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)

TOTAL ROWS: 100

SAMPLE DATA:


SiteID,SiteName,Country,City,StudyID,SiteStatus,ActivationDate,_source_file,_ingested_at
SITE001,Berlin Clinical Research Site 001,Germany,Berlin,ST015,Active,2025-06-09,Site.csv,2026-10-23T15:12:50.732Z
SITE002,Madrid Clinical Research Site 002,Spain,Madrid,ST019,Active,2025-08-24,Site.csv,2026-10-23T15:12:50.732Z
SITE003,Sydney Clinical Research Site 003,Australia,Sydney,ST008,Active,2026-05-22,Site.csv,2026-10-23T15:12:50.732Z
SITE004,Bengaluru Clinical Research Site 004,India,Bengaluru,ST009,Closed,2024-03-31,Site.csv,2026-10-23T15:12:50.732Z
SITE005,New York Clinical Research Site 005,USA,New York,ST019,Active,2024-05-25,Site.csv,2026-10-23T15:12:50.732Z


TABLE: gpo_analytics.bronze.subject

SCHEMA:
root
 |-- SubjectID: string (nullable = true)
 |-- StudyID: string (nullable = true)
 |-- SiteID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Disease: string (nullable = true)
 |-- ScreenDate: string (nullable = true)
 |-- SubjectStatus: string (nullable = true)
 |-- EnrollmentDate: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)

TOTAL ROWS: 3,000

SAMPLE DATA:


SubjectID,StudyID,SiteID,Gender,Age,Disease,ScreenDate,SubjectStatus,EnrollmentDate,_source_file,_ingested_at
SUB00001,ST017,SITE018,Male,57,Inflammatory Disease,2026-01-22,Active,2026-01-26,Subject.csv,2026-10-23T15:12:59.987Z
SUB00002,ST001,SITE030,Female,80,Solid Tumor,2026-09-18,Active,2026-10-01,Subject.csv,2026-10-23T15:12:59.987Z
SUB00003,ST001,SITE045,Unknown,41,Solid Tumor,2025-06-15,Active,2025-06-28,Subject.csv,2026-10-23T15:12:59.987Z
SUB00004,ST001,SITE030,Other,19,Solid Tumor,2023-12-02,Withdrawn,2023-12-23,Subject.csv,2026-10-23T15:12:59.987Z
SUB00005,ST020,SITE042,Male,50,Solid Tumor,2024-12-20,Active,2025-01-06,Subject.csv,2026-10-23T15:12:59.987Z


TABLE: gpo_analytics.bronze.visit

SCHEMA:
root
 |-- VisitID: string (nullable = true)
 |-- SubjectID: string (nullable = true)
 |-- VisitName: string (nullable = true)
 |-- VisitDate: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)

TOTAL ROWS: 14,398

SAMPLE DATA:


VisitID,SubjectID,VisitName,VisitDate,Status,_source_file,_ingested_at
V0000001,SUB00001,Screening,2026-01-22,Completed,Visit.csv,2026-10-23T15:13:04.784Z
V0000002,SUB00001,Baseline,2026-01-29,Cancelled,Visit.csv,2026-10-23T15:13:04.784Z
V0000003,SUB00001,Week 4,2026-02-19,Completed,Visit.csv,2026-10-23T15:13:04.784Z
V0000004,SUB00001,Week 8,2026-03-19,Completed,Visit.csv,2026-10-23T15:13:04.784Z
V0000005,SUB00001,Week 12,2026-04-16,Completed,Visit.csv,2026-10-23T15:13:04.784Z


... (8 outputs left out)

TABLE: gpo_analytics.bronze.labresults

SCHEMA:
root
 |-- LabID: string (nullable = true)
 |-- SubjectID: string (nullable = true)
 |-- TestName: string (nullable = true)
 |-- Result: string (nullable = true)
 |-- Unit: string (nullable = true)
 |-- Flag: string (nullable = true)
 |-- LabDate: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)

TOTAL ROWS: 10,000

SAMPLE DATA:


LabID,SubjectID,TestName,Result,Unit,Flag,LabDate,_source_file,_ingested_at
L0000001,SUB00112,ALT,28.74,U/L,Normal,2026-07-05,LabResults.csv,2026-10-23T15:12:36.667Z
L0000002,SUB02855,Creatinine,0.86,mg/dL,Normal,2026-03-10,LabResults.csv,2026-10-23T15:12:36.667Z
L0000003,SUB01795,ALT,33.37,U/L,Normal,2026-06-06,LabResults.csv,2026-10-23T15:12:36.667Z
L0000004,SUB02738,Creatinine,1.04,mg/dL,Normal,2024-10-09,LabResults.csv,2026-10-23T15:12:36.667Z
L0000005,SUB00057,Hemoglobin,18.96,g/dL,High,2026-07-25,LabResults.csv,2026-10-23T15:12:36.667Z


TABLE: gpo_analytics.bronze.adverseevents

SCHEMA:
root
 |-- AEID: string (nullable = true)
 |-- SubjectID: string (nullable = true)
 |-- Event: string (nullable = true)
 |-- Severity: string (nullable = true)
 |-- Serious: string (nullable = true)
 |-- Relationship: string (nullable = true)
 |-- Outcome: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)

TOTAL ROWS: 1,200

SAMPLE DATA:


AEID,SubjectID,Event,Severity,Serious,Relationship,Outcome,_source_file,_ingested_at
AE000001,SUB00500,Diarrhea,Grade 1,No,Related,Ongoing,AdverseEvents.csv,2026-10-23T15:12:17.228Z
AE000002,SUB01155,Elevated ALT,Grade 2,Yes,Related,Fatal,AdverseEvents.csv,2026-10-23T15:12:17.228Z
AE000003,SUB00424,Diarrhea,Grade 3,Yes,Related,Recovered with Sequelae,AdverseEvents.csv,2026-10-23T15:12:17.228Z
AE000004,SUB01904,Headache,Grade 1,No,Related,Recovered with Sequelae,AdverseEvents.csv,2026-10-23T15:12:17.228Z
AE000005,SUB01758,Headache,Grade 3,Yes,Not Assessed,Recovering,AdverseEvents.csv,2026-10-23T15:12:17.228Z


In [0]:

from pyspark.sql import functions as F

# 1. Specify the catalog and schema containing our Bronze tables.
catalog = "gpo_analytics"
schema = "bronze"

# 2. Define each table and its candidate primary-key column.
table_keys = {
    "study": "StudyID",
    "site": "SiteID",
    "subject": "SubjectID",
    "visit": "VisitID",
    "adverseevents": "AEID",
    "labresults": "LabID",
    "drugdispensation": "DispenseID",
    "randomization": "RandomizationID",
    "shipment": "ShipmentID",
    "investigator": "InvestigatorID"
}

# 3. Store the profiling results in Python lists.
table_summary = []
column_quality = []
key_quality = []

# 4. Process each Bronze table.
for table_name, key_column in table_keys.items():
    full_table_name = f"{catalog}.{schema}.{table_name}"
    df = spark.table(full_table_name)

    # 5. Count the total number of rows.
    total_rows = df.count()
    table_summary.append((table_name, total_rows, len(df.columns)))

    # 6. Check every column for nulls and blank strings.
    for column_name, data_type in df.dtypes:
        is_null = F.col(column_name).isNull()

        if data_type == "string":
            is_blank = F.trim(F.col(column_name)) == ""
            issue_condition = is_null | is_blank
            blank_count = df.filter(is_blank).count()
        else:
            issue_condition = is_null
            blank_count = 0

        null_count = df.filter(is_null).count()
        issue_count = df.filter(issue_condition).count()

        column_quality.append((
            table_name, column_name, null_count,
            blank_count, issue_count
        ))

    # 7. Check for missing and duplicate candidate key values.
    key_null_count = df.filter(
        F.col(key_column).isNull() |
        (F.trim(F.col(key_column)) == "")
    ).count()

    duplicate_groups = (
        df.filter(
            F.col(key_column).isNotNull() &
            (F.trim(F.col(key_column)) != "")
        )
        .groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
    )

    duplicate_extra_rows = (
        duplicate_groups
        .select((F.col("count") - 1).alias("extra_rows"))
        .agg(F.sum("extra_rows").alias("total"))
        .first()["total"]
    ) or 0

    key_quality.append((
        table_name, key_column, key_null_count,
        duplicate_extra_rows
    ))

# 8. Display the table-level summary.
print("TABLE SUMMARY")
display(spark.createDataFrame(
    table_summary, ["table_name", "total_rows", "column_count"]
))

# 9. Display null and blank counts for every column.
print("COLUMN DATA QUALITY")
display(spark.createDataFrame(
    column_quality,
    ["table_name", "column_name", "null_count",
     "blank_count", "null_or_blank_count"]
))

# 10. Display candidate-key quality results.
print("CANDIDATE KEY QUALITY")
display(spark.createDataFrame(
    key_quality,
    ["table_name", "candidate_key", "missing_key_count",
     "duplicate_extra_rows"]
))


TABLE SUMMARY


table_name,total_rows,column_count
study,20,11
site,100,9
subject,3000,11
visit,14398,7
adverseevents,1200,9
labresults,10000,9
drugdispensation,4500,8
randomization,1915,6
shipment,600,10
investigator,100,6


COLUMN DATA QUALITY


table_name,column_name,null_count,blank_count,null_or_blank_count
study,StudyID,0,0,0
study,ProtocolID,0,0,0
study,StudyName,0,0,0
study,Phase,0,0,0
study,TherapeuticArea,0,0,0
study,Status,0,0,0
study,PlannedEnrollment,0,0,0
study,StudyStartDate,0,0,0
study,StudyEndDate,2,0,2
study,_source_file,0,0,0


CANDIDATE KEY QUALITY


table_name,candidate_key,missing_key_count,duplicate_extra_rows
study,StudyID,0,0
site,SiteID,0,0
subject,SubjectID,0,0
visit,VisitID,0,0
adverseevents,AEID,0,0
labresults,LabID,0,0
drugdispensation,DispenseID,0,0
randomization,RandomizationID,0,0
shipment,ShipmentID,0,0
investigator,InvestigatorID,0,0
